This notebook extracts the level and response of every trial that QUEST used, for crowding, acuity and RSVP reading.
It saves two files, so that the fitting notebook (`quest_refit_subsample.ipynb`) can be rerun without extracting again:
- `tidy_both_sessions_trials_level_response.csv`: one row per QUEST update (for RSVP, one row per word).
- `tidy_both_sessions_staircases_quest_params.csv`: one row per staircase, with the QUEST settings and EasyEyes' final estimate (used to check the refit).

`level` is log10 of the threshold parameter: spacingDeg (crowding), targetSizeDeg (acuity), targetDurationSec (RSVP). `response` is 1 for correct, 0 for incorrect.

In [ ]:
import os
import pandas as pd
import numpy as np
import re

In [ ]:
# Set up files
dir_firstSess = 'data/corr_firstSess'
files_firstSess = [f for f in os.listdir(dir_firstSess) if f.endswith('.csv')]

dir_secondSess = 'data/corr_secondSess'
files_secondSess = [f for f in os.listdir(dir_secondSess) if f.endswith('.csv')]

print('Number of total sessions: ', len(files_firstSess) + len(files_secondSess))
print('Number of subjects who completed the second session: ', len(files_secondSess))

### Functions:

In [ ]:
def checkIfComplete(mainOutput):

    complete_val = mainOutput['experimentCompleteBool'].dropna().iloc[0]
    complete_bool = str(complete_val) == 'True'
    assert complete_bool, 'Fatal: experiment not complete!'


def parse_condition(condition_str):
    """
    Parse a condition string of the form <task>_<meridian>_block<repeat>
    Example: 'acuity_R8_block1' -> ('acuity', 'R8', 1)
    """
    match = re.match(r"^(.*?)_(.*?)_block(\d+)$", condition_str)
    if match:
        return match.group(1), match.group(2), int(match.group(3))
    else:
        raise ValueError(f"String '{condition_str}' is not in the expected format.")


def is_true(series):
    # EasyEyes booleans can be read in as bool or as 'TRUE'/'FALSE' strings
    return series.astype(str).str.upper() == 'TRUE'


def first_number(series):
    vals = pd.to_numeric(series, errors='coerce').dropna()
    return vals.iloc[0] if len(vals) else np.nan

In [ ]:
def get_trialwise_level_response(mydir, files, condition_names, session):
    """
    Return two data frames:

    trials: one row per response that QUEST used.
      prolificID, session, conditionName, taskName, meridian, repeat,
      trial_index (1-based QUEST update number), seq_index (1-based row in the staircase; for RSVP one sequence of words),
      word_index (1-3 for RSVP, 1 otherwise), level (log10), response (1 correct, 0 incorrect)

    staircases: one row per staircase, with QUEST settings (tGuess is log10 of thresholdGuess) and
      EasyEyes' final estimate (questMeanAtEndOfTrialsLoop, questSDAtEndOfTrialsLoop).

    Which rows QUEST used: rows with trialGivenToQuest == True, after the last QUEST reset.
    QUEST restarts after practice-until-correct, sometimes without questResetByThresholdPracticeUntilCorrectBool
    being set. So the final run is found from questTrialCountAtEndOfTrial: it starts at the last row whose count
    equals that row's number of responses (1 for letters, 3 for RSVP), and then increases by that number every row.

    RSVP: each row is one sequence of words, all at the same level. QUEST gets one update per word, taken from
    rsvpReadingResponseCorrectBool (e.g. 'TRUE,FALSE,TRUE'). trials.response holds only the last word, so it is
    not used for RSVP.
    """
    all_trials = []
    all_staircases = []

    for fname in files:
        main = pd.read_csv(os.path.join(mydir, fname), low_memory=False)
        checkIfComplete(main)
        prolificID = main['ProlificParticipantID'].dropna().iloc[0]

        for condition_name in condition_names:
            cond = main[main['conditionName'] == condition_name]
            assert not cond.empty, f'Fatal: empty condition, session name: {fname}'
            taskName, meridian, repeat = parse_condition(condition_name)
            responses_per_row = 3 if taskName == 'rsvp' else 1

            # keep the final QUEST run only
            quest_rows = cond[is_true(cond['trialGivenToQuest'])]
            count = quest_rows['questTrialCountAtEndOfTrial'].to_numpy()
            run_start = np.where(count == responses_per_row)[0]
            quest_rows = quest_rows.iloc[run_start[-1]:] if len(run_start) else quest_rows.iloc[0:0]
            assert (np.diff(quest_rows['questTrialCountAtEndOfTrial'].to_numpy()) == responses_per_row).all(), \
                f'Fatal: QUEST count does not increase steadily, session name: {fname}; condition: {condition_name}'

            rows = []
            for seq_index, (_, row) in enumerate(quest_rows.iterrows(), start=1):
                if taskName == 'rsvp':
                    words = [w.strip().upper() == 'TRUE' for w in str(row['rsvpReadingResponseCorrectBool']).split(',')]
                    assert len(words) == responses_per_row, f'Fatal: unexpected number of words, session name: {fname}'
                else:
                    words = [row['trials.response'] == 1]
                for word_index, correct in enumerate(words, start=1):
                    rows.append({'prolificID': prolificID,
                                 'session': session,
                                 'conditionName': condition_name,
                                 'taskName': taskName,
                                 'meridian': meridian,
                                 'repeat': repeat,
                                 'trial_index': len(rows) + 1,
                                 'seq_index': seq_index,
                                 'word_index': word_index,
                                 'level': row['level'],
                                 'response': int(correct)})
            all_trials += rows

            all_staircases.append({'prolificID': prolificID,
                                   'session': session,
                                   'conditionName': condition_name,
                                   'taskName': taskName,
                                   'meridian': meridian,
                                   'repeat': repeat,
                                   'nTrials': len(rows),
                                   'nTrialsLogged': quest_rows['questTrialCountAtEndOfTrial'].iloc[-1] if len(rows) else 0,
                                   'tGuess': first_number(cond['trials.startVal']),
                                   'tGuessSd': first_number(cond['trials.startValSd']),
                                   'pThreshold': first_number(cond['trials.pThreshold']),
                                   'beta': first_number(cond['trials.beta']),
                                   'delta': first_number(cond['trials.delta']),
                                   'gamma': first_number(cond['trials.gamma']),
                                   'grain': first_number(cond['trials.grain']),
                                   'questMeanAtEnd': first_number(cond['questMeanAtEndOfTrialsLoop']),
                                   'questSDAtEnd': first_number(cond['questSDAtEndOfTrialsLoop'])})

    return pd.DataFrame(all_trials), pd.DataFrame(all_staircases)

### Acquire levels and responses, trial wise:
- letter acuity (log deg)
- crowding distance (log deg)
- RSVP reading (word duration, log sec)

In [ ]:
thresholds_names_sess1 = ['crowding_R8_block1', 'crowding_L8_block1',
                          'crowding_R8_block2', 'crowding_L8_block2',
                          'acuity_R8_block1', 'acuity_L8_block1',
                          'rsvp_foveal_block1']
trials_firstSess, staircases_firstSess = get_trialwise_level_response(dir_firstSess, files_firstSess, thresholds_names_sess1, 1)

thresholds_names_sess2 = ['crowding_R8_block3', 'crowding_L8_block3',
                          'crowding_R8_block4', 'crowding_L8_block4',
                          'acuity_R8_block2', 'acuity_L8_block2',
                          'rsvp_foveal_block2']
trials_secondSess, staircases_secondSess = get_trialwise_level_response(dir_secondSess, files_secondSess, thresholds_names_sess2, 2)

In [ ]:
trials = pd.concat([trials_firstSess, trials_secondSess], ignore_index=True)
staircases = pd.concat([staircases_firstSess, staircases_secondSess], ignore_index=True)

# only keep participants who completed both sessions
both_ids = staircases.groupby('prolificID')['session'].nunique()
both_ids = both_ids[both_ids == 2].index
trials = trials[trials['prolificID'].isin(both_ids)].reset_index(drop=True)
staircases = staircases[staircases['prolificID'].isin(both_ids)].reset_index(drop=True)

print(f'Number of participants with both sessions: {len(both_ids)}')

In [ ]:
# checks
assert (staircases['nTrials'] == staircases['nTrialsLogged']).all(), 'Fatal: number of responses does not match QUEST count'
assert set(trials['response'].unique()) <= {0, 1}

# staircases with no QUEST trials (e.g. practice-until-correct never ended); these are dropped in the fitting notebook
empty = staircases[staircases['nTrials'] == 0]
print(f"Warning: {len(empty)} staircases with no QUEST trials, from {empty['prolificID'].nunique()} participant(s)")
print('\nNumber of responses per staircase:')
print(staircases[staircases['nTrials'] > 0].groupby('taskName')['nTrials'].describe())

In [ ]:
# save to csv
trials.to_csv('tidy_both_sessions_trials_level_response.csv', index=False)
staircases.to_csv('tidy_both_sessions_staircases_quest_params.csv', index=False)